In [1]:
# experiments/03_phase3_part3.py
# Goal: freeze remaining 9 layers, measure clean latency for all variants

import torch
import torch._dynamo
import sys
import copy
import statistics
from datasets import load_dataset
from transformers import AutoTokenizer
from torchao.quantization.qat.linear import Int8DynActInt4WeightQATLinear

sys.path.insert(0, '/home/shreya/Coding/optimizer_V2')
from src.graph.boundary_detector import ActivationStabilityDetector
from src.graph.static_converter import (StaticScaleLinear,
                                         convert_stable_layers,
                                         print_conversion_report)

# fix dynamo cache before anything
torch._dynamo.config.cache_size_limit = 64

device    = torch.device('cuda')
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


/home/shreya/venvs/fusion_qat/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ── data ────────────────────────────────────────────────────
dataset_val = load_dataset("glue", "sst2", split="validation")

def encode_and_batch(dataset, batch_size=16):
    sentences = list(dataset['sentence'])
    labels    = list(dataset['label'])
    enc  = tokenizer(sentences, padding='max_length',
                     truncation=True, max_length=64)
    ids   = torch.tensor(enc['input_ids'])
    masks = torch.tensor(enc['attention_mask'])
    labs  = torch.tensor(labels)
    batches = []
    for i in range(0, len(ids) - batch_size, batch_size):
        batches.append({
            'input_ids':      ids[i:i+batch_size],
            'attention_mask': masks[i:i+batch_size],
            'labels':         labs[i:i+batch_size]
        })
    return batches

val_batches = encode_and_batch(dataset_val, batch_size=16)

# ── helpers ──────────────────────────────────────────────────
def measure_accuracy(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_batches:
            inputs = {k: v.to(device) for k, v in batch.items()
                      if k != 'labels'}
            labels = batch['labels'].to(device)
            out    = model(**inputs)
            correct += (out.logits.argmax(-1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total * 100

def measure_latency(model, warmup=10, runs=50):
    compiled = torch.compile(model, backend="inductor")
    model.eval()
    with torch.no_grad():
        for batch in val_batches[:warmup]:
            inputs = {k: v.to(device) for k, v in batch.items()
                      if k != 'labels'}
            compiled(**inputs)
    torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for batch in val_batches[:runs]:
            inputs = {k: v.to(device) for k, v in batch.items()
                      if k != 'labels'}
            start = torch.cuda.Event(enable_timing=True)
            end   = torch.cuda.Event(enable_timing=True)
            start.record()
            compiled(**inputs)
            end.record()
            torch.cuda.synchronize()
            times.append(start.elapsed_time(end))
    return statistics.mean(times), statistics.stdev(times)

def count_layers(model):
    dynamic = sum(1 for _, m in model.named_modules()
                  if isinstance(m, Int8DynActInt4WeightQATLinear))
    static  = sum(1 for _, m in model.named_modules()
                  if isinstance(m, StaticScaleLinear))
    return dynamic, static


In [11]:

# ── load Phase 3 model ───────────────────────────────────────

import os

BASE_DIR = os.path.abspath("..") 

model_path = os.path.join(BASE_DIR, "results", "checkpoints", "phase3_full_model.pt")

model_p3 = torch.load(model_path, map_location=device, weights_only=False)

model_p3 = model_p3.to(device)

dyn, sta = count_layers(model_p3)
print(f"Phase 3 model: {sta} static + {dyn} dynamic")

# ── measure Phase 3 as-is (clean measurement) ───────────────
print("\nMeasuring Phase 3 latency (clean)...")
acc_p3       = measure_accuracy(model_p3)
lat_p3, std_p3 = measure_latency(model_p3)
print(f"Phase 3: {lat_p3:.1f}ms ± {std_p3:.1f}ms  acc={acc_p3:.2f}%")

# ── calibrate remaining dynamic layers ───────────────────────
print("\nCalibrating remaining dynamic layers...")
detector = ActivationStabilityDetector(cv_threshold_static=5.0)
detector.attach_hooks(model_p3)

model_p3.eval()
with torch.no_grad():
    for batch in val_batches:   # use full val set for calibration
        inputs = {k: v.to(device) for k, v in batch.items()
                  if k != 'labels'}
        model_p3(**inputs)

detector.remove_hooks()
remaining_labels = detector.compute_labels()

# show what's left
print(f"\nRemaining dynamic layer stability:")
print(f"{'Layer':<55} {'CV%':>8} {'verdict':>12}")
print("="*80)
for name, info in remaining_labels.items():
    print(f"{name:<55} {info['cv']:>7.2f}%  {info['verdict']:>12}")

# ── combine: Phase 3 training + Phase 2 post-hoc cleanup ────
model_combined = copy.deepcopy(model_p3)
model_combined, report = convert_stable_layers(
    model_combined, remaining_labels, include_borderline=True
)
print_conversion_report(report)

dyn_c, sta_c = count_layers(model_combined)
print(f"Combined model: {sta_c} static + {dyn_c} dynamic")


Phase 3 model: 29 static + 9 dynamic

Measuring Phase 3 latency (clean)...


/home/shreya/venvs/fusion_qat/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0422 03:04:57.358000 64916 torch/_dynamo/convert_frame.py:906] [22/64] torch._dynamo hit config.cache_size_limit (64)
W0422 03:04:57.358000 64916 torch/_dynamo/convert_frame.py:906] [22/64]    function: 'apply_chunking_to_forward' (/home/shreya/venvs/fusion_qat/lib/python3.12/site-packages/transformers/pytorch_utils.py:126)
W0422 03:04:57.358000 64916 torch/_dynamo/convert_frame.py:906] [22/64]    last reason: 22/63: Cache line invalidated because L['forward_fn'] got deallocated
W0422 03:04:57.358000 64916 torch/_dynamo/convert_frame.py:906] [22/64] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0422 03:04:57.358000 64916 torch/_dynamo/convert_frame.py:906] [22/64] To diagnos

Phase 3: 33.4ms ± 72.5ms  acc=87.96%

Calibrating remaining dynamic layers...
Attached hooks to 9 QAT layers

Remaining dynamic layer stability:
Layer                                                        CV%      verdict
distilbert.transformer.layer.0.attention.q_lin             3.38%     STATIC_OK
distilbert.transformer.layer.0.attention.k_lin             3.38%     STATIC_OK
distilbert.transformer.layer.0.attention.v_lin             3.38%     STATIC_OK
distilbert.transformer.layer.2.attention.out_lin           2.91%     STATIC_OK
distilbert.transformer.layer.3.attention.out_lin           2.64%     STATIC_OK
distilbert.transformer.layer.5.attention.out_lin           3.78%     STATIC_OK
distilbert.transformer.layer.5.ffn.lin1                    5.20%    BORDERLINE
distilbert.transformer.layer.5.ffn.lin2                    6.02%    BORDERLINE
pre_classifier                                             5.60%    BORDERLINE

CONVERSION REPORT
  Converted to static: 9
  Kept dynamic:       

In [16]:

# ── measure combined ─────────────────────────────────────────
print("\nMeasuring combined model...")
acc_combined         = measure_accuracy(model_combined)
lat_combined, std_combined = measure_latency(model_combined)
print(f"Combined: {lat_combined:.1f}ms ± {std_combined:.1f}ms  "
      f"acc={acc_combined:.2f}%")

# ── final table ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  COMPLETE RESULTS TABLE")
print(f"{'='*65}")
print(f"  {'Model':<35} {'Latency':>9}  {'Accuracy':>10}  {'Frozen':>8}")
print(f"  {'-'*60}")
print(f"  {'FP32 baseline':<35} {'12.0ms':>9}  {'—':>10}  {'—':>8}")
print(f"  {'QAT dynamic':<35} {'34.8ms':>9}  {'87.38%':>10}  {'0/38':>8}")
print(f"  {'Phase 2 post-hoc static':<35} {'12.0ms':>9}  {'86.69%':>10}  {'34/38':>8}")
print(f"  {'Phase 3 training-aware':<35} {f'{lat_p3:.1f}ms':>9}  "
      f"{f'{acc_p3:.2f}%':>10}  {'29/38':>8}")
print(f"  {'Phase 3 + cleanup':<35} {f'{lat_combined:.1f}ms':>9}  "
      f"{f'{acc_combined:.2f}%':>10}  {f'{sta_c}/38':>8}")
print(f"  {'-'*60}")
print(f"  FP32 reference latency: 12.0ms")
print(f"{'='*65}")

BASE_DIR = os.path.abspath("..")  # go from experiments → root

save_dir = os.path.join(BASE_DIR, "results", "checkpoints")
save_path = os.path.join(save_dir, "phase3_combined_full.pt")

torch.save(model_combined, save_path)

print("\nSaved:", save_path)



Measuring combined model...
Combined: 17.1ms ± 0.2ms  acc=86.46%

  COMPLETE RESULTS TABLE
  Model                                 Latency    Accuracy    Frozen
  ------------------------------------------------------------
  FP32 baseline                          12.0ms           —         —
  QAT dynamic                            34.8ms      87.38%      0/38
  Phase 2 post-hoc static                12.0ms      86.69%     34/38
  Phase 3 training-aware                 33.4ms      87.96%     29/38
  Phase 3 + cleanup                      17.1ms      86.46%     38/38
  ------------------------------------------------------------
  FP32 reference latency: 12.0ms

Saved: /home/shreya/Coding/optimizer_V2/results/checkpoints/phase3_combined_full.pt
